In [1]:
from src.load_data import load_radioml
X, mods, snrs = load_radioml("../data/RML2016.10a_dict.pkl")
print(X.shape)
print(sorted(set(mods)))
print(sorted(set(snrs)))


(220000, 2, 128)
[np.str_('8PSK'), np.str_('AM-DSB'), np.str_('AM-SSB'), np.str_('BPSK'), np.str_('CPFSK'), np.str_('GFSK'), np.str_('PAM4'), np.str_('QAM16'), np.str_('QAM64'), np.str_('QPSK'), np.str_('WBFM')]
[np.int64(-20), np.int64(-18), np.int64(-16), np.int64(-14), np.int64(-12), np.int64(-10), np.int64(-8), np.int64(-6), np.int64(-4), np.int64(-2), np.int64(0), np.int64(2), np.int64(4), np.int64(6), np.int64(8), np.int64(10), np.int64(12), np.int64(14), np.int64(16), np.int64(18)]


In [2]:
import numpy as np
from src.features import amplitude_features

indices = np.where((mods == "BPSK") & (snrs == 18))[0]

amplitude_features(X[indices][0])

{'amplitude_std': np.float32(0.01713086),
 'amplitude_skew': np.float32(2.5270596),
 'amplitude_kurtosis': np.float32(5.053034),
 'amplitude_peak_ratio': np.float32(9.079748)}

In [3]:
from src.features import amplitude_features

indices = np.where((mods == "AM-DSB") & (snrs == 18))[0]

amplitude_features(X[indices][0])

{'amplitude_std': np.float32(6.256845e-05),
 'amplitude_skew': np.float32(-0.060612556),
 'amplitude_kurtosis': np.float32(0.059408426),
 'amplitude_peak_ratio': np.float32(1.0232768)}

In [7]:
from src.features import amplitude_features

means = [amplitude_features(X[i])["amplitude_std"] for i in range(0, 220000, 1000)]
print(min(means), max(means))

6.256845e-05 0.024198266


Amplitude_mean is constant (roughly 1/128) across the dataset because RadioML normalizes each signal, so I removed it.

In [5]:
from src.features import extract_all_features

features_df = extract_all_features(X)
features_df["mod"] = mods
features_df["snr"] = snrs

features_df.to_csv("../data/features.csv", index=False)
features_df.head()

,amplitude_std,amplitude_skew,amplitude_kurtosis,amplitude_peak_ratio,phase_mean,phase_std,phase_skew,phase_kurtosis,frequency_mean,frequency_std,frequency_skew,frequency_kurtosis,mod,snr
0,0.002740,-0.332200,0.558705,2.108140,-0.011607,1.778630,-0.094661,-1.368191,0.023482,0.772706,-0.129574,4.050705,QPSK,2
1,0.002668,0.025988,-0.042563,1.877517,-0.148191,1.684268,-0.089195,-1.160113,0.101569,0.776628,-0.805517,3.629322,QPSK,2
2,0.002873,0.115225,-0.526414,1.867858,0.155166,1.709790,-0.106533,-1.115362,0.072386,0.606628,0.788918,3.988327,QPSK,2
3,0.002991,0.079101,-0.680335,1.881248,-0.007726,1.743644,-0.052037,-1.219978,0.003475,0.574669,1.501462,6.934639,QPSK,2
4,0.002578,-0.056228,-0.535420,1.745609,-0.070726,1.585114,0.353147,-0.938411,0.004503,0.527505,-0.312477,2.574277,QPSK,2


In [8]:
from src.features import amplitude_features, phase_features, frequency_features

for i in range(5):
    mine = {**amplitude_features(X[i]), **phase_features(X[i]), **frequency_features(X[i])}
    for name, value in mine.items():
        if not np.isclose(value, features_df.loc[i, name], rtol=1e-4):
            print("Mismatch:", i, name, value, features_df.loc[i, name])
print("Check done")

Check done
